In [1]:
# Build a training loop for a customized fully connected network
# for multi-class classification problem
import torch
import torch.nn as nn

class FullyConnectedNetwork(nn.Module):
    def __init__(self, d_in, d_ff, n_class):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_in, d_ff),
            nn.Mish(),
            nn.Linear(d_ff, n_class)
        )
    
    def forward(self, x):
        x = self.fc(x)
        return x

In [2]:
# Build network
d_in = 10
d_ff = 256
n_class = 3
fcn = FullyConnectedNetwork(d_in, d_ff, n_class)

In [16]:
# Set up random data
n_sample = 500
batch_size = 32
x = torch.rand(n_sample, d_in)
y = torch.randint(0, n_class, (n_sample,))

In [45]:
x.shape

torch.Size([500, 10])

In [47]:
y.shape

torch.Size([500])

In [ ]:
dataset = torch.utils.data.TensorDataset(x, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [70]:
# implement my own tensordataset
class TensorDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        assert len(x) == len(y), "dimension mismatch for x and y!"
        self.x = x
        self.y = y
    
    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [63]:
dataset = TensorDataset(x, y)

In [64]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [65]:
# loss function and optimizer
learning_rate = 1e-3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fcn.parameters(), lr=learning_rate)

In [66]:
# Training loop
num_epoch = 100

for epoch in range(num_epoch):
    running_loss = 0.0

    for inputs, label in dataloader:
        optimizer.zero_grad()
        loss = loss_fn(fcn(inputs), label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'{epoch+1}/{num_epoch}: Loss: {running_loss/len(dataloader):.4f}')

10/100: Loss: 1.0597
20/100: Loss: 1.0594
30/100: Loss: 1.0575
40/100: Loss: 1.0520
50/100: Loss: 1.0474
60/100: Loss: 1.0482
70/100: Loss: 1.0407
80/100: Loss: 1.0365
90/100: Loss: 1.0366
100/100: Loss: 1.0347


In [67]:
# Inference
n_test = 20
x_test = torch.rand(n_test, d_in)

with torch.no_grad():
    outputs = fcn(x_test)
    # _, X_predicted = torch.max(outputs.data, 1)
    # print(f'Predictions: {X_predicted}')
    

In [68]:
_, x_pred_class = torch.max(outputs, dim=-1)

In [44]:
x_pred_class

tensor([2, 0, 2, 2, 2, 1, 2, 1, 1, 2, 1, 2, 2, 2, 2, 1, 1, 2, 2, 2])

In [ ]:
# imporve by AI
import torch
import torch.nn as nn

torch.manual_seed(0)

class FullyConnectedNetwork(nn.Module):
    def __init__(self, d_in, d_ff, n_class):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_in, d_ff),
            nn.Mish(),
            nn.Linear(d_ff, n_class)
        )

    def forward(self, x):
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d_in, d_ff, n_class = 10, 256, 3
fcn = FullyConnectedNetwork(d_in, d_ff, n_class).to(device)

n_sample, batch_size = 500, 32
x = torch.rand(n_sample, d_in)
y = torch.randint(0, n_class, (n_sample,))
dataset = torch.utils.data.TensorDataset(x, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fcn.parameters(), lr=1e-3)

num_epoch = 20
for epoch in range(num_epoch):
    fcn.train()
    running_loss = 0.0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = fcn(inputs)
        loss = loss_fn(logits, labels)

        loss.backward()
        # torch.nn.utils.clip_grad_norm_(fcn.parameters(), 1.0)  # optional
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"{epoch+1}/{num_epoch}: Loss={running_loss/len(dataloader):.4f}")


In [30]:
torch.cuda.is_available()

False